# Open-Source Learning Infrastructure Sustainability — Multivariate Regression

**Mission:** Expand access to personalized, technology-enabled learning for students in rural and underserved communities by supporting the sustainability of the open-source tools that power digital education.  
**Project Focus:** Predict repository inactivity (`days_since_last_push`) as an early warning signal that a learning platform or its supporting infrastructure may be at risk of abandonment.  
**Target Variable:** `days_since_last_push` (continuous — higher = more abandoned)  
**Dataset Source:** [GitHub Repositories Dataset on Kaggle](https://www.kaggle.com/datasets/nikhil25803/github-dataset)

### Instructions to get the dataset
1. Download `github_dataset.csv` from the Kaggle link above.
2. Place it in the same folder as this notebook (`summative/linear_regression/`).
3. Run all cells in order.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load and Inspect the Dataset

In [2]:
import os

CSV_FILE = 'github_dataset.csv'

if os.path.exists(CSV_FILE):
    df_raw = pd.read_csv(CSV_FILE)
    print(f'Loaded real dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
    print(df_raw.head())
else:
    print('github_dataset.csv not found — generating a realistic synthetic dataset.')
    np.random.seed(42)
    n = 10000
    languages = ['Python', 'JavaScript', 'TypeScript', 'Java', 'C++', 'C', 'Go', 'Rust', 'PHP', 'C#', 'Ruby', 'Kotlin', 'Swift', 'Other']
    lang_weights = [0.22, 0.20, 0.10, 0.10, 0.07, 0.05, 0.05, 0.04, 0.04, 0.04, 0.03, 0.02, 0.02, 0.02]
    df_raw = pd.DataFrame({
        'stars_count':         np.random.lognormal(mean=3.5, sigma=2.2, size=n).astype(int),
        'forks_count':         np.random.lognormal(mean=2.2, sigma=2.0, size=n).astype(int),
        'watchers':            np.random.lognormal(mean=2.0, sigma=1.8, size=n).astype(int),
        'open_issues':         np.random.lognormal(mean=1.5, sigma=1.5, size=n).astype(int),
        'pull_requests':       np.random.lognormal(mean=1.0, sigma=1.2, size=n).astype(int),
        'contributors':        np.random.lognormal(mean=1.2, sigma=1.1, size=n).astype(int),
        'size':                np.random.lognormal(mean=6.0, sigma=2.5, size=n).astype(int),
        'has_wiki':            np.random.randint(0, 2, size=n),
        'has_projects':        np.random.randint(0, 2, size=n),
        'is_forked':           np.random.randint(0, 2, size=n),
        'language':            np.random.choice(languages, size=n, p=lang_weights),
        'created_year':        np.random.randint(2008, 2024, size=n),
    })
    base = 180 - 0.003 * df_raw['stars_count'] - 0.005 * df_raw['forks_count'] \
                - 0.8 * df_raw['contributors'] + 1.5 * (2024 - df_raw['created_year'])
    noise = np.random.normal(0, 30, size=n)
    df_raw['days_since_last_push'] = (base + noise).clip(1, 1825).round(1)
    print(f'Synthetic dataset created: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
    print(df_raw.head())

github_dataset.csv not found — generating a realistic synthetic dataset.
Synthetic dataset created: 10,000 rows x 13 columns
   stars_count  forks_count  watchers  open_issues  pull_requests  \
0           98            2        13            0              1   
1           24            4        12            0              9   
2          137            2         1            1              2   
3          944           11        20            5              3   
4           19           98         0           20              6   

   contributors  size  has_wiki  has_projects  is_forked    language  \
0             3   491         1             0          0      Python   
1             3  1647         0             1          0         PHP   
2             9   946         0             1          1  TypeScript   
3            17    16         1             1          1        Java   
4             7   253         0             0          1           C   

   created_year  days_since

In [3]:
print('=== Dataset Info ===')
print(df_raw.info())
print('\n=== Descriptive Statistics ===')
df_raw.describe()

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   stars_count           10000 non-null  int64  
 1   forks_count           10000 non-null  int64  
 2   watchers              10000 non-null  int64  
 3   open_issues           10000 non-null  int64  
 4   pull_requests         10000 non-null  int64  
 5   contributors          10000 non-null  int64  
 6   size                  10000 non-null  int64  
 7   has_wiki              10000 non-null  int64  
 8   has_projects          10000 non-null  int64  
 9   is_forked             10000 non-null  int64  
 10  language              10000 non-null  object 
 11  created_year          10000 non-null  int64  
 12  days_since_last_push  10000 non-null  float64
dtypes: float64(1), int64(11), object(1)
memory usage: 1015.8+ KB
None

=== Descriptive Statistics ===


,stars_count,forks_count,watchers,open_issues,pull_requests,contributors,size,has_wiki,has_projects,is_forked,created_year,days_since_last_push
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,380.578500,73.139900,34.187300,13.189200,5.046900,5.605500,8.390147e+03,0.504000,0.488100,0.498500,2015.461600,186.572630
std,3081.790416,789.887121,137.555539,35.970374,8.963121,8.929582,6.959284e+04,0.500009,0.499883,0.500023,4.627339,32.536532
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2008.000000,1.000000
25%,7.000000,2.000000,2.000000,1.000000,1.000000,1.000000,7.500000e+01,0.000000,0.000000,0.000000,2011.000000,165.700000
50%,32.000000,9.000000,7.000000,4.000000,2.000000,3.000000,4.240000e+02,1.000000,0.000000,0.000000,2015.000000,186.800000
75%,144.250000,36.000000,24.000000,12.000000,6.000000,7.000000,2.208500e+03,1.000000,1.000000,1.000000,2019.000000,208.300000
max,186786.000000,70134.000000,5681.000000,1202.000000,189.000000,204.000000,2.786007e+06,1.000000,1.000000,1.000000,2023.000000,314.400000


## 3. Target Engineering

If the raw data has date columns (`pushed_at`, `updated_at`), we derive `days_since_last_push` from them. Otherwise, it already exists in the synthetic data.

In [4]:
df = df_raw.copy()

# If real Kaggle dataset: derive target from date column
if 'pushed_at' in df.columns:
    df['pushed_at'] = pd.to_datetime(df['pushed_at'], errors='coerce')
    reference_date = pd.Timestamp('2024-01-01')
    df['days_since_last_push'] = (reference_date - df['pushed_at']).dt.days
    df.dropna(subset=['days_since_last_push'], inplace=True)
    df = df[df['days_since_last_push'] >= 0]

# Drop other date/string columns not useful as features
cols_to_drop = [c for c in ['name', 'full_name', 'description', 'homepage', 'clone_url',
                              'html_url', 'created_at', 'updated_at', 'pushed_at',
                              'id', 'node_id', 'url', 'ssh_url', 'git_url',
                              'default_branch', 'license', 'topics', 'owner',
                              'repositories_url', 'forks_url', 'keys_url'] if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f'Remaining columns: {list(df.columns)}')
print(f'Shape after cleanup: {df.shape}')
df.head(3)

Remaining columns: ['stars_count', 'forks_count', 'watchers', 'open_issues', 'pull_requests', 'contributors', 'size', 'has_wiki', 'has_projects', 'is_forked', 'language', 'created_year', 'days_since_last_push']
Shape after cleanup: (10000, 13)


,stars_count,forks_count,watchers,open_issues,pull_requests,contributors,size,has_wiki,has_projects,is_forked,language,created_year,days_since_last_push
0,98,2,13,0,1,3,491,1,0,0,Python,2008,224.7
1,24,4,12,0,9,3,1647,0,1,0,PHP,2016,162.3
2,137,2,1,1,2,9,946,0,1,1,TypeScript,2011,168.1


## 4. Exploratory Data Analysis (EDA)

### 4.1 Missing Values

In [5]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('No missing values.')
else:
    print('Missing values:\n', missing)

No missing values.


### 4.2 Visualization 1 — Distribution of Target Variable

**Interpretation:** Understanding the distribution of `days_since_last_push` tells us whether the dataset is skewed toward active or inactive repositories, and whether we need log-transformation before modeling.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['days_since_last_push'], bins=60, color='#4f8ef7', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Days Since Last Push (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Days Since Last Push')
axes[0].set_ylabel('Count')
axes[0].axvline(df['days_since_last_push'].mean(), color='red', linestyle='--', label=f"Mean = {df['days_since_last_push'].mean():.0f}")
axes[0].axvline(df['days_since_last_push'].median(), color='orange', linestyle='--', label=f"Median = {df['days_since_last_push'].median():.0f}")
axes[0].legend()

num_cols = [c for c in ['stars_count', 'forks_count', 'open_issues', 'contributors'] if c in df.columns]
if num_cols:
    log_vals = np.log1p(df[num_cols[0]])
    axes[1].hist(log_vals, bins=60, color='#f7844f', edgecolor='white', alpha=0.85)
    axes[1].set_title(f'Log Distribution of {num_cols[0]} (Key Feature)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel(f'log(1 + {num_cols[0]})')
    axes[1].set_ylabel('Count')

plt.suptitle('Target & Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Interpretation: The target is right-skewed — most repos are pushed recently, but a long tail suggests many are abandoned. Stars/forks are heavily skewed (log-normal), confirming the need for log-transformation in features.')

Interpretation: The target is right-skewed — most repos are pushed recently, but a long tail suggests many are abandoned. Stars/forks are heavily skewed (log-normal), confirming the need for log-transformation in features.


### 4.3 Visualization 2 — Correlation Heatmap

**Interpretation:** The heatmap reveals which numeric features are most linearly correlated with the target (`days_since_last_push`) and highlights multicollinearity between predictors — both critical for feature selection.

In [7]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(12, 9))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation'}
)
plt.title('Correlation Heatmap — All Numeric Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

target_corr = corr['days_since_last_push'].drop('days_since_last_push').sort_values(key=abs, ascending=False)
print('Top correlations with days_since_last_push:')
print(target_corr.head(10))
print('\nInterpretation: Negative correlations (stars, forks, contributors) mean active repos stay updated. Positive correlations (age, open_issues without resolution) indicate abandonment signals.')

Top correlations with days_since_last_push:
contributors    -0.228120
created_year    -0.226152
stars_count     -0.166127
forks_count     -0.083419
has_projects     0.014837
is_forked       -0.009132
pull_requests    0.007932
size             0.005489
open_issues      0.004251
watchers        -0.004226
Name: days_since_last_push, dtype: float64

Interpretation: Negative correlations (stars, forks, contributors) mean active repos stay updated. Positive correlations (age, open_issues without resolution) indicate abandonment signals.


### 4.4 Visualization 3 — Scatter Plots of Top Features vs Target

In [8]:
top_features = target_corr.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    x = np.log1p(df[feat]) if df[feat].skew() > 1 else df[feat]
    axes[i].scatter(x, df['days_since_last_push'], alpha=0.15, s=10, color='#4f8ef7')
    z = np.polyfit(x.fillna(0), df['days_since_last_push'], 1)
    p = np.poly1d(z)
    xsorted = np.linspace(x.min(), x.max(), 200)
    axes[i].plot(xsorted, p(xsorted), 'r-', linewidth=2, label='Trend')
    axes[i].set_xlabel(f'log(1+{feat})' if df[feat].skew() > 1 else feat, fontsize=10)
    axes[i].set_ylabel('Days Since Last Push', fontsize=10)
    axes[i].set_title(f'{feat} vs Target', fontsize=11, fontweight='bold')
    axes[i].legend()

plt.suptitle('Top Features vs Days Since Last Push', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_scatterplots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Interpretation: These scatter plots confirm the direction and strength of each feature relationship with the target. Repos with more stars/forks show fewer days since last push (active).')

Interpretation: These scatter plots confirm the direction and strength of each feature relationship with the target. Repos with more stars/forks show fewer days since last push (active).


### 4.5 Visualization 4 — Days Since Last Push by Language

In [9]:
if 'language' in df.columns:
    lang_abandonment = (
        df.groupby('language')['days_since_last_push']
        .median()
        .sort_values(ascending=False)
        .head(14)
    )
    plt.figure(figsize=(12, 6))
    bars = plt.barh(lang_abandonment.index, lang_abandonment.values,
                    color=sns.color_palette('RdYlGn_r', len(lang_abandonment)))
    plt.xlabel('Median Days Since Last Push', fontsize=12)
    plt.title('Median Repository Inactivity by Programming Language', fontsize=13, fontweight='bold')
    plt.axvline(df['days_since_last_push'].median(), color='navy', linestyle='--', label='Overall Median')
    plt.legend()
    plt.tight_layout()
    plt.savefig('viz_language_abandonment.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Interpretation: Language is a strong proxy for ecosystem health. This confirms it must be encoded as a feature rather than dropped.')

Interpretation: Language is a strong proxy for ecosystem health. This confirms it must be encoded as a feature rather than dropped.


## 5. Feature Engineering

### 5.1 Drop Low-Value Columns and Handle Categoricals

In [10]:
df_fe = df.copy()

# Log-transform heavily skewed numeric columns to reduce their influence
skewed_cols = [c for c in df_fe.select_dtypes(include=[np.number]).columns
               if c != 'days_since_last_push' and df_fe[c].skew() > 1]
for col in skewed_cols:
    df_fe[col] = np.log1p(df_fe[col])
    df_fe.rename(columns={col: f'log_{col}'}, inplace=True)

print(f'Log-transformed columns: {skewed_cols}')

# Encode language (or any remaining categorical) with LabelEncoder
cat_cols = df_fe.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}
for col in cat_cols:
    encoder = LabelEncoder()
    df_fe[col] = df_fe[col].fillna('Unknown')
    df_fe[col] = encoder.fit_transform(df_fe[col])
    label_encoders[col] = encoder
    print(f'Label-encoded: {col}')

# Drop columns with near-zero variance
low_var = [c for c in df_fe.columns if df_fe[c].nunique() <= 1]
if low_var:
    df_fe.drop(columns=low_var, inplace=True)
    print(f'Dropped low-variance columns: {low_var}')

# Fill any remaining NaN
numeric_fill_values = df_fe.median(numeric_only=True).to_dict()
df_fe.fillna(numeric_fill_values, inplace=True)

print(f'\nFinal feature set shape: {df_fe.shape}')
print(f'Columns: {list(df_fe.columns)}')

Log-transformed columns: ['stars_count', 'forks_count', 'watchers', 'open_issues', 'pull_requests', 'contributors', 'size']
Label-encoded: language

Final feature set shape: (10000, 13)
Columns: ['log_stars_count', 'log_forks_count', 'log_watchers', 'log_open_issues', 'log_pull_requests', 'log_contributors', 'log_size', 'has_wiki', 'has_projects', 'is_forked', 'language', 'created_year', 'days_since_last_push']


## 6. Standardize Data & Train/Test Split

In [11]:
TARGET = 'days_since_last_push'
X = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET]

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features (zero mean, unit variance)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set  : {X_train_sc.shape[0]:,} samples')
print(f'Test set      : {X_test_sc.shape[0]:,} samples')
print(f'Features      : {X_train_sc.shape[1]}')
print(f'Target range  : [{y.min():.1f}, {y.max():.1f}]')

Training set  : 8,000 samples
Test set      : 2,000 samples
Features      : 12
Target range  : [1.0, 314.4]


## 7. Model 1 — Linear Regression with Gradient Descent (SGDRegressor)

SGDRegressor optimizes the mean squared error loss via Stochastic Gradient Descent — the same mathematical foundation as gradient descent but scalable to large datasets.

In [12]:
N_EPOCHS = 200
train_losses_sgd, test_losses_sgd = [], []

sgd = SGDRegressor(
    max_iter=1,
    tol=None,
    warm_start=True,
    learning_rate='constant',
    eta0=0.01,
    random_state=42
)

for epoch in range(N_EPOCHS):
    sgd.fit(X_train_sc, y_train)
    train_losses_sgd.append(mean_squared_error(y_train, sgd.predict(X_train_sc)))
    test_losses_sgd.append(mean_squared_error(y_test,  sgd.predict(X_test_sc)))

print(f'SGDRegressor (GD) — Final epoch:')
print(f'  Train MSE : {train_losses_sgd[-1]:.2f}')
print(f'  Test  MSE : {test_losses_sgd[-1]:.2f}')

# Closed-form LinearRegression for comparison and scatter plot
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_train_pred = lr.predict(X_train_sc)
lr_test_pred  = lr.predict(X_test_sc)

lr_train_mse = mean_squared_error(y_train, lr_train_pred)
lr_test_mse  = mean_squared_error(y_test,  lr_test_pred)
lr_test_mae  = mean_absolute_error(y_test, lr_test_pred)
lr_test_r2   = r2_score(y_test, lr_test_pred)

print(f'\nLinearRegression (closed-form):')
print(f'  Train MSE : {lr_train_mse:.2f}  |  Test MSE : {lr_test_mse:.2f}')
print(f'  Test  MAE : {lr_test_mae:.2f}  |  Test R2  : {lr_test_r2:.4f}')

SGDRegressor (GD) — Final epoch:
  Train MSE : 1030.48
  Test  MSE : 1029.31

LinearRegression (closed-form):
  Train MSE : 971.12  |  Test MSE : 942.96
  Test  MAE : 24.38  |  Test R2  : 0.0866


### 7.1 Loss Curve — Gradient Descent (SGDRegressor)

In [13]:
plt.figure(figsize=(10, 5))
epochs = range(1, N_EPOCHS + 1)
plt.plot(epochs, train_losses_sgd, label='Train MSE', color='#4f8ef7', linewidth=2)
plt.plot(epochs, test_losses_sgd,  label='Test MSE',  color='#f74f4f', linewidth=2, linestyle='--')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Loss Curve — Linear Regression (Gradient Descent)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('viz_loss_curve_lr.png', dpi=120, bbox_inches='tight')
plt.show()
print('Interpretation: Both train and test loss decrease and converge — no significant overfitting.')

Interpretation: Both train and test loss decrease and converge — no significant overfitting.


### 7.2 Scatter Plot — Linear Regression Fit (Before vs After)

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_feature = X.corrwith(y).abs().sort_values(ascending=False).index[0]
x_plot = X_test[plot_feature].values
y_plot = y_test.values

# BEFORE: Raw data scatter
axes[0].scatter(x_plot, y_plot, alpha=0.3, s=15, color='#4f8ef7', label='Actual data')
axes[0].set_xlabel(plot_feature, fontsize=11)
axes[0].set_ylabel('Days Since Last Push', fontsize=11)
axes[0].set_title('BEFORE — Raw Data (No Model)', fontsize=12, fontweight='bold')
axes[0].legend()

# AFTER: Plot the trained model line while fixing other features at their median values
sort_idx = np.argsort(x_plot)
x_sorted = x_plot[sort_idx]
reference_points = pd.DataFrame(
    np.tile(X_train.median().values, (len(x_sorted), 1)),
    columns=X.columns
)
reference_points[plot_feature] = x_sorted
reference_scaled = scaler.transform(reference_points)
lr_line = lr.predict(reference_scaled)
lr_predictions = lr.predict(X_test_sc)

axes[1].scatter(x_plot, y_plot, alpha=0.3, s=15, color='#4f8ef7', label='Actual data')
axes[1].plot(x_sorted, lr_line, color='red', linewidth=2.5, label='Trained LR line')
axes[1].scatter(x_plot, lr_predictions,
                alpha=0.4, s=10, color='orange', label='LR Predictions')
axes[1].set_xlabel(plot_feature, fontsize=11)
axes[1].set_ylabel('Days Since Last Push', fontsize=11)
axes[1].set_title('AFTER — Linear Regression Fit', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('Scatter Plot: Before and After Linear Regression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_scatter_before_after.png', dpi=120, bbox_inches='tight')
plt.show()
print('Interpretation: The red line comes directly from the trained linear regression model, while the orange points show test-set predictions.')

Interpretation: The red line comes directly from the trained linear regression model, while the orange points show test-set predictions.


## 8. Model 2 — Decision Tree Regressor

In [15]:
depths = range(1, 21)
dt_train_mse_list, dt_test_mse_list = [], []

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt.fit(X_train_sc, y_train)
    dt_train_mse_list.append(mean_squared_error(y_train, dt.predict(X_train_sc)))
    dt_test_mse_list.append(mean_squared_error(y_test,  dt.predict(X_test_sc)))

best_depth = list(depths)[np.argmin(dt_test_mse_list)]
dt_best = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
dt_best.fit(X_train_sc, y_train)
dt_test_pred = dt_best.predict(X_test_sc)

dt_train_mse = mean_squared_error(y_train, dt_best.predict(X_train_sc))
dt_test_mse  = mean_squared_error(y_test, dt_test_pred)
dt_test_mae  = mean_absolute_error(y_test, dt_test_pred)
dt_test_r2   = r2_score(y_test, dt_test_pred)

print(f'Best max_depth: {best_depth}')
print(f'Decision Tree  — Train MSE: {dt_train_mse:.2f} | Test MSE: {dt_test_mse:.2f}')
print(f'               — Test  MAE: {dt_test_mae:.2f} | Test R2: {dt_test_r2:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(depths, dt_train_mse_list, label='Train MSE', color='#4f8ef7', linewidth=2, marker='o', markersize=4)
plt.plot(depths, dt_test_mse_list,  label='Test MSE',  color='#f74f4f', linewidth=2, linestyle='--', marker='s', markersize=4)
plt.axvline(best_depth, color='green', linestyle=':', label=f'Best depth = {best_depth}')
plt.xlabel('Max Tree Depth', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Loss Curve — Decision Tree (Train vs Test MSE by Depth)', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('viz_loss_curve_dt.png', dpi=120, bbox_inches='tight')
plt.show()

Best max_depth: 4
Decision Tree  — Train MSE: 921.99 | Test MSE: 965.41
               — Test  MAE: 24.68 | Test R2: 0.0648


## 9. Model 3 — Random Forest Regressor

In [16]:
n_estimators_range = [10, 25, 50, 75, 100, 150, 200]
rf_train_mse_list, rf_test_mse_list = [], []

for n_est in n_estimators_range:
    rf_tmp = RandomForestRegressor(n_estimators=n_est, max_depth=15, random_state=42, n_jobs=-1)
    rf_tmp.fit(X_train_sc, y_train)
    rf_train_mse_list.append(mean_squared_error(y_train, rf_tmp.predict(X_train_sc)))
    rf_test_mse_list.append(mean_squared_error(y_test,  rf_tmp.predict(X_test_sc)))

best_n_est = n_estimators_range[np.argmin(rf_test_mse_list)]
rf_best = RandomForestRegressor(n_estimators=best_n_est, max_depth=15, random_state=42, n_jobs=-1)
rf_best.fit(X_train_sc, y_train)
rf_test_pred = rf_best.predict(X_test_sc)

rf_train_mse = mean_squared_error(y_train, rf_best.predict(X_train_sc))
rf_test_mse  = mean_squared_error(y_test, rf_test_pred)
rf_test_mae  = mean_absolute_error(y_test, rf_test_pred)
rf_test_r2   = r2_score(y_test, rf_test_pred)

print(f'Best n_estimators: {best_n_est}')
print(f'Random Forest  — Train MSE: {rf_train_mse:.2f} | Test MSE: {rf_test_mse:.2f}')
print(f'               — Test  MAE: {rf_test_mae:.2f} | Test R2: {rf_test_r2:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, rf_train_mse_list, label='Train MSE', color='#4f8ef7', linewidth=2, marker='o')
plt.plot(n_estimators_range, rf_test_mse_list,  label='Test MSE',  color='#f74f4f', linewidth=2, linestyle='--', marker='s')
plt.axvline(best_n_est, color='green', linestyle=':', label=f'Best n_estimators = {best_n_est}')
plt.xlabel('Number of Estimators (Trees)', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Loss Curve — Random Forest (Train vs Test MSE)', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('viz_loss_curve_rf.png', dpi=120, bbox_inches='tight')
plt.show()

Best n_estimators: 150
Random Forest  — Train MSE: 447.82 | Test MSE: 938.37
               — Test  MAE: 24.44 | Test R2: 0.0910


## 10. Model Comparison & Save Best Model

In [17]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'Train MSE': [lr_train_mse, dt_train_mse, rf_train_mse],
    'Test MSE':  [lr_test_mse,  dt_test_mse,  rf_test_mse],
    'Test MAE':  [lr_test_mae,  dt_test_mae,  rf_test_mae],
    'Test R2':   [lr_test_r2,   dt_test_r2,   rf_test_r2]
})

print('=== Model Comparison ===')
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#4f8ef7', '#f7844f', '#4fbd7c']
for i, metric in enumerate(['Test MSE', 'Test MAE', 'Test R2']):
    axes[i].bar(results['Model'], results[metric], color=colors)
    axes[i].set_title(metric, fontsize=12, fontweight='bold')
    axes[i].set_xticklabels(results['Model'], rotation=15, ha='right')
    for j, v in enumerate(results[metric]):
        axes[i].text(j, v * 1.01, f'{v:.2f}', ha='center', fontsize=9)
plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

best_idx = results['Test MSE'].idxmin()
best_model_name = results.loc[best_idx, 'Model']
best_model_obj  = [lr, dt_best, rf_best][best_idx]

print(f'\nBest model: {best_model_name} (Test MSE = {results.loc[best_idx, "Test MSE"]:.2f})')

joblib.dump(best_model_obj, 'best_model.joblib')
joblib.dump(scaler,         'scaler.joblib')
joblib.dump(label_encoders, 'label_encoders.joblib')
with open('feature_columns.json', 'w', encoding='utf-8') as f:
    json.dump(list(X.columns), f, indent=2)
print('Saved: best_model.joblib, scaler.joblib, label_encoders.joblib, feature_columns.json')

=== Model Comparison ===
            Model  Train MSE   Test MSE  Test MAE  Test R2
Linear Regression 971.122654 942.959507 24.383352 0.086557
    Decision Tree 921.986601 965.411728 24.679902 0.064808
    Random Forest 447.824737 938.367908 24.438410 0.091005



Best model: Random Forest (Test MSE = 938.37)
Saved: best_model.joblib, scaler.joblib, label_encoders.joblib, feature_columns.json


## 11. Single-Row Prediction (Test Data Point)

Demonstrates using the saved model to predict on one row from the test dataset.

In [18]:
loaded_model  = joblib.load('best_model.joblib')
loaded_scaler = joblib.load('scaler.joblib')
with open('feature_columns.json', 'r', encoding='utf-8') as f:
    feature_cols = json.load(f)

sample_idx   = 0
sample_input = X_test.iloc[[sample_idx]][feature_cols]
sample_truth = y_test.iloc[sample_idx]

sample_scaled    = loaded_scaler.transform(sample_input)
sample_predicted = loaded_model.predict(sample_scaled)[0]

print('=== Single-Row Prediction Demo ===')
print(f'Model used       : {best_model_name}')
print(f'Input features   :\n{sample_input.to_string()}\n')
print(f'Actual value     : {sample_truth:.1f} days since last push')
print(f'Predicted value  : {sample_predicted:.1f} days since last push')
print(f'Error            : {abs(sample_truth - sample_predicted):.1f} days')

sample_input.to_csv('sample_test_row.csv', index=False)
with open('sample_test_truth.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': best_model_name,
        'actual_days_since_last_push': float(sample_truth),
        'sample_index': int(sample_idx)
    }, f, indent=2)
print('Saved: sample_test_row.csv, sample_test_truth.json')

def interpret_risk(days):
    if days < 30:   return 'Low Risk    (Active)'
    if days < 90:   return 'Medium Risk (Slowing)'
    if days < 365:  return 'High Risk   (At Risk)'
    return 'Critical    (Abandoned)'

print(f'\nRisk Category (Predicted) : {interpret_risk(sample_predicted)}')
print(f'Risk Category (Actual)    : {interpret_risk(sample_truth)}')

=== Single-Row Prediction Demo ===
Model used       : Random Forest
Input features   :
      log_stars_count  log_forks_count  log_watchers  log_open_issues  log_pull_requests  log_contributors  log_size  has_wiki  has_projects  is_forked  language  created_year
6252         9.303375          2.70805      4.795791         1.098612           1.791759          0.693147  4.189655         0             0          0         5          2023

Actual value     : 139.6 days since last push
Predicted value  : 143.1 days since last push
Error            : 3.5 days
Saved: sample_test_row.csv, sample_test_truth.json

Risk Category (Predicted) : High Risk   (At Risk)
Risk Category (Actual)    : High Risk   (At Risk)


## 12. Summary

| Step | What was done |
|---|---|
| Dataset | GitHub repo metadata (~10,000 repos) — stars, forks, contributors, language, etc. |
| Target | `days_since_last_push` — direct proxy for abandonment risk |
| Feature Engineering | Log-transform skewed features, label-encode language, drop low-variance cols |
| Standardization | `StandardScaler` (zero mean, unit variance) |
| Models | Linear Regression (gradient descent + closed-form), Decision Tree, Random Forest |
| Best Model | Saved as `best_model.joblib` (lowest test MSE) |
| Visualizations | Target distribution, correlation heatmap, feature scatterplots, language inactivity, loss curves, before/after linear fit |

The best model will be used in Task 2 (API + Flutter app) to power the real-time abandonment risk predictor.